# Embedding-Based Customer Support Insight Platform

This notebook uses a lightweight modern NLP pipeline for 200k support tickets:

```text
Support Tickets
    -> Lightweight Preprocessing
    -> SentenceTransformer Embeddings
    -> FAISS Similarity Search
    -> Response Recommendation
    -> KMeans Issue Clustering
    -> Frustration and Business Insights
```


## 1. Data Preparation

Input columns used by the project:

- Ticket Subject
- Ticket Description
- Resolution
- Ticket Type
- Ticket Priority
- Customer Satisfaction Rating


In [ ]:
import os
import re
import string
import warnings

import requests

import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 140)

RANDOM_STATE = 42
DATA_PATH = "customer_support_tickets_200k.csv"
MAX_ROWS = 200_000



In [ ]:
raw_df = pd.read_csv(DATA_PATH, nrows=MAX_ROWS)
print(raw_df.shape)
raw_df.head()


In [ ]:
# The 200k dataset uses snake_case names. Rename them into the assignment column names.
column_map = {
    "ticket_id": "Ticket ID",
    "category": "Ticket Type",
    "issue_description": "Ticket Description",
    "resolution_notes": "Resolution",
    "priority": "Ticket Priority",
    "customer_satisfaction_score": "Customer Satisfaction Rating",
    "ticket_created_date": "Ticket Created Date",
    "ticket_resolved_date": "Ticket Resolved Date",
    "status": "Ticket Status",
    "product": "Product Purchased"
}
work = raw_df.rename(columns=column_map).copy()

# Create a short subject from the category and the beginning of the issue text.
work["Ticket Subject"] = (
    work["Ticket Type"].fillna("Support issue").astype(str)
    + " - "
    + work["Ticket Description"].fillna("").astype(str).str.slice(0, 80)
)

input_columns = [
    "Ticket Subject",
    "Ticket Description",
    "Resolution",
    "Ticket Type",
    "Ticket Priority",
    "Customer Satisfaction Rating"
]

work = work[input_columns + [
    "Ticket ID",
    "Ticket Created Date",
    "Ticket Resolved Date",
    "Ticket Status",
    "Product Purchased",
    "resolution_time_hours",
    "escalated",
    "sla_breached"
]].copy()

# Convert numeric columns before any groupby/mean operations.
work["Customer Satisfaction Rating"] = pd.to_numeric(
    work["Customer Satisfaction Rating"],
    errors="coerce"
)
work["resolution_time_hours"] = pd.to_numeric(
    work["resolution_time_hours"],
    errors="coerce"
)

# Convert Yes/No columns to 1/0 so escalation and SLA rates can be averaged.
for column in ["escalated", "sla_breached"]:
    work[column] = (
        work[column]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({"yes": 1, "no": 0, "true": 1, "false": 0, "1": 1, "0": 0})
        .fillna(0)
        .astype(int)
    )

print(work.shape)
work[input_columns + ["resolution_time_hours", "escalated", "sla_breached"]].head()


## 2. Text Construction

The main semantic feature is:

```python
ticket_text = Ticket Subject + Ticket Description
```


In [ ]:
work["ticket_text"] = (
    work["Ticket Subject"].fillna("").astype(str)
    + " "
    + work["Ticket Description"].fillna("").astype(str)
)

work[["Ticket Subject", "Ticket Description", "ticket_text"]].head()


## 3. Text Preprocessing

Use light preprocessing only:

```text
lowercase -> remove punctuation/special characters -> lemmatization
```

No aggressive stopword removal is used because embedding models already understand context.


In [ ]:
import nltk
from nltk.stem import WordNetLemmatizer

# Run once if WordNet is missing:
# nltk.download("wordnet")
# nltk.download("omw-1.4")

lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return " ".join(tokens)

work["clean_ticket_text"] = work["ticket_text"].apply(clean_text)
work[["ticket_text", "clean_ticket_text"]].head()


## 4. Embedding Generation

Generate dense semantic embeddings using Hugging Face SentenceTransformers.

Recommended model: `all-MiniLM-L6-v2`

Each ticket becomes a 384-dimensional semantic vector.

Install once if needed:

```python
%pip install sentence-transformers
```


In [ ]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

texts = work["clean_ticket_text"].tolist()

ticket_embeddings = embedding_model.encode(
    texts,
    batch_size=256,
    show_progress_bar=True,
    normalize_embeddings=True
)

ticket_embeddings = np.asarray(ticket_embeddings, dtype="float32")
print(ticket_embeddings.shape)


## 5. Vector Storage With FAISS

FAISS stores the 384-dimensional vectors for fast semantic search.

Install once if needed:

```python
%pip install faiss-cpu
```


In [ ]:
import faiss

embedding_dim = ticket_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(embedding_dim)
faiss_index.add(ticket_embeddings)

print("Vectors stored:", faiss_index.ntotal)
print("Embedding dimensions:", embedding_dim)


## 6. Similar Ticket Retrieval

Pipeline:

```text
New Ticket -> Embedding -> FAISS Search -> Top-K Similar Tickets
```


In [ ]:
def embed_new_ticket(subject, description):
    text = clean_text(str(subject) + " " + str(description))
    embedding = embedding_model.encode(
        [text],
        normalize_embeddings=True
    )
    return np.asarray(embedding, dtype="float32")


def find_similar_tickets(subject, description, top_k=5):
    query_embedding = embed_new_ticket(subject, description)
    scores, indices = faiss_index.search(query_embedding, top_k)

    results = work.iloc[indices[0]].copy()
    results["similarity_score"] = scores[0]
    return results[[
        "Ticket ID",
        "Ticket Subject",
        "Ticket Type",
        "Ticket Priority",
        "Resolution",
        "similarity_score"
    ]]

find_similar_tickets(
    "refund not received",
    "customer says money never came back after refund was approved",
    top_k=5
)


## Hugging Face LLM Setup

Set your Hugging Face token outside the notebook when possible:

```python
import os
os.environ["HF_API_TOKEN"] = "your_token_here"
```

The notebook reads the token from `HF_API_TOKEN`, so the key is not hardcoded.


In [ ]:
HF_API_TOKEN = os.getenv("HF_API_TOKEN")
HF_LLM_MODEL = "google/flan-t5-large"
HF_API_URL = f"https://api-inference.huggingface.co/models/{HF_LLM_MODEL}"


def call_hf_llm(prompt, max_new_tokens=180):
    if not HF_API_TOKEN:
        return "HF_API_TOKEN is not set. Add your Hugging Face token to the environment and rerun this cell."

    headers = {"Authorization": f"Bearer {HF_API_TOKEN}"}
    payload = {
        "inputs": prompt,
        "parameters": {
            "max_new_tokens": max_new_tokens,
            "temperature": 0.2,
            "return_full_text": False
        }
    }

    response = requests.post(HF_API_URL, headers=headers, json=payload, timeout=60)
    response.raise_for_status()
    result = response.json()

    if isinstance(result, list) and len(result) > 0:
        return result[0].get("generated_text", str(result[0])).strip()

    if isinstance(result, dict):
        return result.get("generated_text", str(result)).strip()

    return str(result).strip()


## 7. Suggested Response Recommendation

Use retrieved tickets and their previous resolutions as a lightweight retrieval-based support assistant.


In [ ]:
def recommend_response(subject, description, top_k=5):
    similar = find_similar_tickets(subject, description, top_k=top_k)
    resolved = similar[
        similar["Resolution"].notna()
        & (similar["Resolution"].astype(str).str.len() > 0)
    ]

    resolution_examples = "\n".join(
        f"- {text}"
        for text in resolved["Resolution"].astype(str).head(3)
    )

    prompt = f"""
You are a customer support assistant.

New ticket subject: {subject}
New ticket description: {description}

Similar past resolutions:
{resolution_examples}

Write one concise, helpful suggested response for the support agent.
"""

    suggestion = call_hf_llm(prompt, max_new_tokens=180)

    return {
        "suggested_response": suggestion,
        "similar_tickets": similar
    }

example_response = recommend_response(
    "refund not received",
    "customer says money never came back after refund was approved",
    top_k=5
)

print(example_response["suggested_response"])
example_response["similar_tickets"]


## 8. Recurring Issue Detection

Cluster embeddings to identify common customer problems.

This notebook uses KMeans because it is simple and scalable for 200k rows.


In [ ]:
N_CLUSTERS = 12

cluster_model = KMeans(
    n_clusters=N_CLUSTERS,
    random_state=RANDOM_STATE,
    n_init=10
)

work["issue_cluster"] = cluster_model.fit_predict(ticket_embeddings)

vectorizer = CountVectorizer(max_features=3000, stop_words="english", ngram_range=(1, 2))
term_matrix = vectorizer.fit_transform(work["clean_ticket_text"])
terms = np.array(vectorizer.get_feature_names_out())


def top_terms_for_cluster(cluster_id, top_n=8):
    mask = work["issue_cluster"].values == cluster_id
    mean_terms = np.asarray(term_matrix[mask].mean(axis=0)).ravel()
    top_indices = mean_terms.argsort()[::-1][:top_n]
    return ", ".join(terms[top_indices])

cluster_summary = (
    work.groupby("issue_cluster")
    .agg(
        ticket_count=("Ticket ID", "count"),
        top_ticket_type=("Ticket Type", lambda x: x.mode().iat[0]),
        avg_satisfaction=("Customer Satisfaction Rating", "mean"),
        avg_resolution_hours=("resolution_time_hours", "mean"),
        escalation_rate=("escalated", "mean"),
        sla_breach_rate=("sla_breached", "mean")
    )
    .reset_index()
)

cluster_summary["top_terms"] = cluster_summary["issue_cluster"].apply(top_terms_for_cluster)
cluster_summary.sort_values("ticket_count", ascending=False).head(12)


## 9. Sentiment / Frustration Analysis

Use satisfaction ratings and operational signals to identify frustrated customers, escalation risk, and churn indicators.


In [ ]:
def llm_sentiment_for_ticket(ticket_text, rating, priority):
    prompt = f"""
Classify this support ticket into exactly one sentiment label: negative, neutral, or positive.
Also give one short reason.

Ticket: {ticket_text}
Customer satisfaction rating: {rating}
Priority: {priority}

Return format:
label: <negative|neutral|positive>
reason: <short reason>
"""
    return call_hf_llm(prompt, max_new_tokens=80)

# Keep scalable numeric features for all 200k tickets.
# The LLM is used for interpretation/explanation instead of hardcoded text labels per row.
work["priority_score"] = pd.to_numeric(
    work["Ticket Priority"].map({"Low": 1, "Medium": 2, "High": 3, "Critical": 4}),
    errors="coerce"
).fillna(2)

work["frustration_score"] = (
    (5 - work["Customer Satisfaction Rating"].fillna(3)) * 2
    + work["priority_score"]
    + work["escalated"].astype(int) * 2
    + work["sla_breached"].astype(int) * 2
)

# Use rank-based levels instead of fixed hardcoded bins.
work["frustration_level"] = pd.qcut(
    work["frustration_score"].rank(method="first"),
    q=3,
    labels=["low", "medium", "high"]
)

sample_for_llm = work.head(5).copy()
sample_for_llm["llm_sentiment_analysis"] = sample_for_llm.apply(
    lambda row: llm_sentiment_for_ticket(
        row["clean_ticket_text"],
        row["Customer Satisfaction Rating"],
        row["Ticket Priority"]
    ),
    axis=1
)

sample_for_llm[[
    "Ticket ID",
    "Ticket Type",
    "Ticket Priority",
    "Customer Satisfaction Rating",
    "frustration_score",
    "frustration_level",
    "llm_sentiment_analysis"
]]


## 10. Business Insight Generation

Generate insights for:

- most common issue
- increasing complaint cluster
- high-frustration categories
- slowest resolutions


In [ ]:
work["Ticket Created Date"] = pd.to_datetime(work["Ticket Created Date"], errors="coerce")
work["month"] = work["Ticket Created Date"].dt.to_period("M").astype(str)

most_common_issues = cluster_summary.sort_values("ticket_count", ascending=False)

high_frustration_categories = (
    work.groupby("Ticket Type")
    .agg(
        tickets=("Ticket ID", "count"),
        avg_frustration=("frustration_score", "mean"),
        high_frustration_rate=("frustration_level", lambda x: (x == "high").mean()),
        avg_satisfaction=("Customer Satisfaction Rating", "mean")
    )
    .reset_index()
    .sort_values("avg_frustration", ascending=False)
)

slowest_resolutions = (
    work.groupby("Ticket Type")
    .agg(
        tickets=("Ticket ID", "count"),
        avg_resolution_hours=("resolution_time_hours", "mean"),
        sla_breach_rate=("sla_breached", "mean")
    )
    .reset_index()
    .sort_values("avg_resolution_hours", ascending=False)
)

monthly_cluster_counts = (
    work.groupby(["month", "issue_cluster"])
    .size()
    .reset_index(name="tickets")
    .sort_values(["issue_cluster", "month"])
)
monthly_cluster_counts["previous_month_tickets"] = monthly_cluster_counts.groupby("issue_cluster")["tickets"].shift(1)
monthly_cluster_counts["mom_change"] = monthly_cluster_counts["tickets"] - monthly_cluster_counts["previous_month_tickets"]

latest_month = monthly_cluster_counts["month"].max()
increasing_complaint_clusters = (
    monthly_cluster_counts[monthly_cluster_counts["month"] == latest_month]
    .merge(cluster_summary, on="issue_cluster", how="left")
    .sort_values("mom_change", ascending=False)
)

insight_context = f"""
Most common issue clusters:
{most_common_issues.head(5).to_string(index=False)}

Increasing complaint clusters:
{increasing_complaint_clusters.head(5).to_string(index=False)}

High-frustration categories:
{high_frustration_categories.head(5).to_string(index=False)}

Slowest resolution categories:
{slowest_resolutions.head(5).to_string(index=False)}
"""

business_insight_prompt = f"""
You are a support operations analyst.
Use the tables below to write 5 concise business insights.
For each insight include the business value or action.

{insight_context}
"""

llm_business_insights = call_hf_llm(business_insight_prompt, max_new_tokens=250)

print(llm_business_insights)

print("Most common issues")
display(most_common_issues.head(10))

print("Increasing complaint clusters")
display(increasing_complaint_clusters.head(10))

print("High-frustration categories")
display(high_frustration_categories.head(10))

print("Slowest resolutions")
display(slowest_resolutions.head(10))


## Export Outputs


In [ ]:
ticket_predictions = work[[
    "Ticket ID",
    "Ticket Subject",
    "Ticket Type",
    "Ticket Priority",
    "Customer Satisfaction Rating",
    "sentiment",
    "frustration_level",
    "issue_cluster"
]]

ticket_predictions.to_csv("ticket_predictions.csv", index=False)
cluster_summary.to_csv("recurring_issue_clusters.csv", index=False)
high_frustration_categories.to_csv("high_frustration_categories.csv", index=False)
slowest_resolutions.to_csv("slowest_resolutions.csv", index=False)
increasing_complaint_clusters.to_csv("increasing_complaint_clusters.csv", index=False)

print("Saved outputs successfully.")
